# 04 — Evaluation
Loads the trained model, runs test-set evaluation, confusion matrix, and Grad-CAM++ visualisation.
6-class: Bacterial_Blight, Blast, Brown_Spot, Tungro, Healthy_Rice_Leaf, Hispa.

## Imports & config

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from safetensors.torch import load_file
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import random

PROCESSED_DIR = Path('../data/processed')
MODEL_PATH    = Path('../models/resnet50_rice.safetensors')
OUTPUT_DIR    = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES      = ['Bacterial_Blight', 'Blast', 'Brown_Spot', 'Tungro', 'Healthy_Rice_Leaf', 'Hispa']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(CLASSES)
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 4
MEAN         = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD          = np.array([0.229, 0.224, 0.225], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
if device.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Classes : {CLASSES}')


## Model definition & load

In [ ]:
class RiceResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=False):
        super().__init__()
        weights       = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone      = models.resnet50(weights=weights)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.head     = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(2048, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


model = RiceResNet(num_classes=NUM_CLASSES, pretrained=False).to(device)
model.load_state_dict(load_file(str(MODEL_PATH)))
model.eval()
print(f'Model loaded — {NUM_CLASSES} classes')


## Preprocessing & dataset

In [ ]:
def preprocess(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    return (img - MEAN) / STD

def to_tensor(img):
    return torch.from_numpy(np.transpose(img, (2, 0, 1)).copy())


class RiceLeafDataset(Dataset):
    def __init__(self, root):
        self.samples = []
        for cls in CLASSES:
            cls_dir = Path(root) / cls
            if not cls_dir.exists():
                print(f'  [WARNING] Missing folder: {cls_dir}')
                continue
            for ext in ('*.jpg', '*.jpeg', '*.JPG', '*.JPEG', '*.png'):
                for p in cls_dir.glob(ext):
                    self.samples.append((p, CLASS_TO_IDX[cls]))
        if not self.samples:
            raise RuntimeError(f'No images found under {root}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(str(path))
        if img is None:
            raise RuntimeError(f'Failed to read: {path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return to_tensor(preprocess(img)), label


test_ds     = RiceLeafDataset(PROCESSED_DIR / 'test')
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

from collections import Counter
counts = Counter(label for _, label in test_ds.samples)
print(f'\n{"Class":<22} {"Count":>8}')
print('-' * 32)
for i, cls in enumerate(CLASSES):
    print(f'{cls:<22} {counts[i]:>8}')
print('-' * 32)
print(f'{"TOTAL":<22} {len(test_ds):>8}')


## Run predictions on test set

In [ ]:
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(device, non_blocking=True)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
print(f'Predictions complete — {len(all_preds)} samples')


## Classification report

In [ ]:
acc = accuracy_score(all_labels, all_preds)
print(f'Overall Accuracy: {acc * 100:.2f}%\n')

unique_labels   = np.unique(all_labels)
present_classes = [CLASSES[i] for i in unique_labels]
print(f'Classes in test set: {present_classes}\n')
print(classification_report(all_labels, all_preds,
                             labels=unique_labels,
                             target_names=present_classes,
                             digits=4))


## Confusion matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].tick_params(axis='x', rotation=30)
axes[0].tick_params(axis='y', rotation=0)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalized)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].tick_params(axis='x', rotation=30)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()


## Grad-CAM++ setup

In [ ]:
class GradCAMPlusPlus:
    """
    Grad-CAM++ — squares the gradients before global averaging,
    producing tighter, more precise heatmaps than standard Grad-CAM.
    Especially useful for small lesions (Blast, Brown_Spot, Hispa).
    Hooks layer4 (7x7 spatial grid, highest semantic level).
    """
    def __init__(self, model):
        self.model       = model
        self.activations = None
        self.gradients   = None
        target = list(model.backbone.children())[7]   # layer4
        target.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        target.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, tensor):
        self.model.eval()
        tensor = tensor.unsqueeze(0).to(device)
        logits = self.model(tensor)
        probs  = F.softmax(logits, dim=1)
        pred   = logits.argmax(1).item()
        conf   = probs[0, pred].item()

        self.model.zero_grad()
        logits[0, pred].backward()

        # Grad-CAM++: square gradients before spatial averaging → tighter localisation
        weights = (self.gradients ** 2).mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * self.activations).sum(dim=1)).squeeze()
        cam     = cam.cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam     = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam, pred, conf


def overlay_heatmap(bgr, cam, alpha=0.45):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    return cv2.addWeighted(bgr, 1 - alpha, heatmap, alpha, 0)


gradcam = GradCAMPlusPlus(model)
print('Grad-CAM++ ready')


## Grad-CAM++ sample predictions (1 per class)

In [ ]:
sample_paths = []
for cls in CLASSES:
    cls_dir = PROCESSED_DIR / 'test' / cls
    if not cls_dir.exists():
        print(f'  [WARNING] Missing test folder: {cls_dir}')
        continue
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    if imgs:
        sample_paths.append((random.choice(imgs), CLASS_TO_IDX[cls]))

fig, axes = plt.subplots(len(sample_paths), 3,
                         figsize=(12, 4 * len(sample_paths)))

for row, (img_path, true_label) in enumerate(sample_paths):
    raw     = cv2.imread(str(img_path))
    disp    = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))
    rgb     = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
    tensor  = to_tensor(preprocess(rgb))

    cam, pred, conf = gradcam.generate(tensor)
    overlay = overlay_heatmap(disp, cam)

    axes[row, 0].imshow(cv2.cvtColor(disp, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f'Original\nTrue: {CLASSES[true_label]}', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(
        f'Grad-CAM++ Overlay\nPred: {CLASSES[pred]} ({conf*100:.1f}%)', fontsize=9)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(cam, cmap='jet')
    axes[row, 2].set_title('Heatmap only', fontsize=9)
    axes[row, 2].axis('off')

plt.suptitle('Grad-CAM++ Sample Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gradcam_samples.png', dpi=150)
plt.show()


## Single image prediction with Grad-CAM++

In [ ]:
def predict_image(img_path, model, gradcam, show=True, save_path=None):
    """
    Predict a single image and display Grad-CAM++ heatmap.
    Returns: (pred_class_name, confidence, cam_numpy)
    """
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        raise ValueError(f'Could not read image: {img_path}')

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tensor  = to_tensor(preprocess(img_rgb))

    cam, pred_idx, conf = gradcam.generate(tensor)

    disp    = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    overlay = overlay_heatmap(disp, cam, alpha=0.45)

    print(f'Predicted : {CLASSES[pred_idx]}')
    print(f'Confidence: {conf*100:.2f}%')

    if show:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(cv2.cvtColor(disp,    cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original');        axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Grad-CAM++ — {CLASSES[pred_idx]} ({conf*100:.1f}%)')
        axes[1].axis('off')
        axes[2].imshow(cam, cmap='jet')
        axes[2].set_title('Heatmap'); axes[2].axis('off')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150)
        plt.show()

    return CLASSES[pred_idx], conf, cam

# Usage:
# predict_image('/path/to/leaf.jpg', model, gradcam)
